In [1]:
# Notebooks live in Notebooks/, but every data path is relative to the repo root.
import os, sys
while not os.path.isdir("data") and os.getcwd() != "/":
    os.chdir("..")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

cwd: /home/user1/WhatDoLLMsWant


# Books pilot — does the new domain work end to end? (2026-10-12)

Two qwen-7B runs on `books_small` (14 books: the 2 most famous per tag), frame `reading`,
no contract and `fantasy=yes`. Design: [books_domain_design.ipynb](books_domain_design.ipynb).

What to look at, in order:

1. **Sanity** — row counts, saturation, positional bias.
2. **Per-book ranking** (position-corrected Borda) — does it look like a preference at all,
   and do the 5 templates agree?
3. **Model vs Goodreads** — rank correlation with average rating, with ratings count, with year.
4. **Tag weights** — `FeatureBT` on the 8 hidden tags only (title / author / year dropped).
5. **The contract** — do fantasy books rise under "I prefer fantasy."? Adherence on the pairs
   where exactly one book is fantasy.

`DATA_DIR` can point at synthetic runs (`scratchpad/fake/books_pilot`, made by `fake_pilot.py` with
known weights: classics +2, fantasy +1, romance −1, contract +3 on fantasy, slot A −1.5) to check the
notebook itself.

In [2]:
import json, glob
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from src.bt import FeatureBT
import src.alternatives as alt
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

DATA_DIR = os.environ.get("BOOKS_PILOT_DIR", "data/books_pilot")
TAGS = alt.book_tags
meta = pd.read_csv("src/books/books.csv").set_index("title")[["author", "year", "average_rating", "ratings_count", "pop_rank_all"]]

runs = {}
for cfg_path in sorted(glob.glob(f"{DATA_DIR}/*/config.json")):
    d = os.path.dirname(cfg_path)
    if not os.path.isfile(f"{d}/scores.csv"):          # incomplete run (config written before the loop)
        print("skip incomplete:", d); continue
    cfg = json.load(open(cfg_path))
    key = (cfg["model_family"], cfg["model_size"], cfg["frame"]["name"], cfg["constraints_id"])
    assert key not in runs, f"duplicate run for {key}"
    runs[key] = pd.read_csv(f"{d}/scores.csv")
    print(key, "<-", os.path.basename(d), "| rows:", len(runs[key]), "| prefix:", repr(cfg["prompt_prefix"]))

('qwen', '7', 'reading', 'none') <- 1447314 | rows: 910 | prefix: 'I am looking for a book to read.'
('qwen', '7', 'reading', 'fantasy=yes') <- 1447317 | rows: 910 | prefix: 'I am looking for a book to read. I prefer fantasy.'


## 1. Sanity

In [3]:
def sanity(df):
    n = df.a_title.nunique(); T = df.template_idx.nunique()
    m = df.score_a - df.score_b
    return dict(books=n, templates=T, rows=len(df), expected=n * (n - 1) * T,
                sat_winner=float((np.maximum(df.score_a, df.score_b) > -0.1).mean()),   # winner logprob ~ 0
                slotA_wins=float((m > 0).mean()), mean_margin=float(m.mean()))
pd.DataFrame({k: sanity(df) for k, df in runs.items()}).T

books  templates   rows  expected  sat_winner  slotA_wins  mean_margin
qwen 7 reading none          14.0        5.0  910.0     910.0    0.636264    0.517582    -0.585944
               fantasy=yes   14.0        5.0  910.0     910.0    0.831868    0.351648    -3.477850

`slotA_wins` far from 0.5 = positional bias (expected; qwen-7B favours slot B on laptops).
`mean_margin` is the raw γ estimate (every pair appears in both orders, so content cancels).

## 2. Per-book ranking: position-corrected Borda

For each unordered pair, $D = (m(x,y) - m(y,x))/2$ is immune to the slot bias and to any constant.
A book's score is its mean $D$ against the others; the rank is over the 14 books.

In [4]:
def book_scores(df):
    """mean position-corrected D per book, per template and pooled"""
    m = df.assign(m=df.score_a - df.score_b)
    fwd = m[["template_idx", "a_title", "b_title", "m"]]
    rev = m[["template_idx", "a_title", "b_title", "m"]].rename(columns={"a_title": "b_title", "b_title": "a_title"})
    rev = rev.assign(m=-rev.m)
    D = pd.concat([fwd, rev]).groupby(["template_idx", "a_title", "b_title"]).m.mean().reset_index()   # D(x,y)
    per_t = D.groupby(["a_title", "template_idx"]).m.mean().unstack()
    per_t["pooled"] = per_t.mean(axis=1)
    return per_t.sort_values("pooled", ascending=False)

BOOK = {b["title"]: b for b in alt.books}
tables = {k: book_scores(df) for k, df in runs.items()}
for k, t in tables.items():
    print("====", k)
    show = t[["pooled"]].join(meta).copy()
    show["tags"] = [", ".join(tg for tg in TAGS if BOOK[ti][tg] == "yes") for ti in show.index]
    show["rank"] = range(1, len(show) + 1)
    print(show.round(2).to_string())
    rho = t.drop(columns="pooled").corr(method="spearman")
    print("template agreement (Spearman of book ranks): mean off-diagonal = %.2f, min = %.2f" %
          (rho.values[np.triu_indices_from(rho, 1)].mean(), rho.values[np.triu_indices_from(rho, 1)].min()))

==== ('qwen', '7', 'reading', 'none')
                                       pooled               author  year  average_rating  ratings_count  pop_rank_all                           tags  rank
a_title                                                                                                                                                   
Harry Potter and the Sorcerer's Stone    5.03         J.K. Rowling  1997            4.44        4602479             2           fantasy, young-adult     1
To Kill a Mockingbird                    4.19           Harper Lee  1960            4.25        3198671             4                       classics     2
The Hunger Games                         3.46      Suzanne Collins  2008            4.34        4780653             1   science-fiction, young-adult     3
1984                                     3.45        George Orwell  1949            4.14        1956832            12      science-fiction, classics     4
The Fault in Our Stars          

## 3. Model vs Goodreads

Does the model like what readers rate highly, what is most read, or what is old?

In [5]:
rows = []
for k, t in tables.items():
    j = t[["pooled"]].join(meta)
    rows.append({"run": k[-1], **{c: spearmanr(j.pooled, j[c])[0] for c in ["average_rating", "ratings_count", "year"]}})
pd.DataFrame(rows).set_index("run").round(2)

,average_rating,ratings_count,year
run,,,
none,0.67,0.31,-0.00
fantasy=yes,0.03,0.48,0.48


## 4. Tag weights

`FeatureBT` on the 8 tag columns only. A weight is `yes − no` for that tag (the fit centres each
feature to mean 0, so the difference is what is identified). R² on 14 books with 8 binary tags will be
modest: the tags do not explain book identity.

**Pilot caveat:** with 14 books and 8 tags the weights are confounded (classics sits on 8 of the 14).
On the synthetic runs the fit got the contract (+3.06 vs +3 true) and the sign of fantasy / romance
right, but put −1.7 on mystery-thriller whose true weight is 0. Read the per-book ranking and the
contract check from the pilot; read tag weights only from the full 48.

In [6]:
def tag_bt(df):
    cols = ["score_a", "score_b"] + [f"a_{t}" for t in TAGS] + [f"b_{t}" for t in TAGS]
    bt = FeatureBT(pmi_c=0.0, r2_warn=None).fit(df[cols])
    w = {t: bt.weights_[t]["yes"] - bt.weights_[t]["no"] for t in TAGS}
    return w, bt
W = {}
for k, df in runs.items():
    w, bt = tag_bt(df); W[k[-1]] = w
    print(k[-1], "| R2 = %.3f | gamma = %.2f" % (bt.r2_, bt.gamma_))
pd.DataFrame(W).round(2)

none | R2 = 0.203 | gamma = -0.59
fantasy=yes | R2 = 0.777 | gamma = -3.48


,none,fantasy=yes
fantasy,-0.20,9.36
science-fiction,4.87,1.14
romance,3.92,-2.00
mystery-thriller,-0.51,5.28
young-adult,-0.61,11.43
classics,1.15,3.26
historical-fiction,2.55,1.94
nonfiction,0.20,-5.09


## 5. The contract: "I prefer fantasy."

- **Δ per book** = pooled score under the contract − without. Fantasy books should rise, the rest fall
  (the scores are zero-sum across books, so only the relative picture is meaningful).
- **Adherence** = on the pairs where exactly one book is fantasy, the share where the fantasy book wins
  (D > 0), without and with the contract.

In [7]:
base = next((k for k in runs if k[-1] == "none"), None)
con = next((k for k in runs if k[-1] == "fantasy=yes"), None)
if base and con:
    d = (tables[con].pooled - tables[base].pooled).rename("delta").to_frame()
    d["fantasy"] = [BOOK[ti]["fantasy"] for ti in d.index]
    print(d.sort_values("delta", ascending=False).round(2).to_string())
    print("\nmean delta: fantasy %.2f | other %.2f" % (d[d.fantasy == "yes"].delta.mean(), d[d.fantasy == "no"].delta.mean()))

    def adherence(df):
        m = df.assign(m=df.score_a - df.score_b)
        one = m[m.a_fantasy != m.b_fantasy]
        fwd = one[["template_idx", "a_title", "b_title", "m", "a_fantasy"]]
        rev = fwd.rename(columns={"a_title": "b_title", "b_title": "a_title"}).assign(m=-fwd.m, a_fantasy=np.where(fwd.a_fantasy == "yes", "no", "yes"))
        D = pd.concat([fwd, rev]).groupby(["template_idx", "a_title", "b_title"]).agg(m=("m", "mean"), f=("a_fantasy", "first")).reset_index()
        D = D[D.f == "yes"]
        return float((D.m > 0).mean()), float(D.m.mean())
    for k in (base, con):
        wr, g = adherence(runs[k]); print(f"{k[-1]:12} fantasy wins {wr:.0%} of mixed pairs, mean D = {g:+.2f}")
else:
    print("need both runs for the contract check; have:", [k[-1] for k in runs])

                                       delta fantasy
a_title                                             
Twilight                               21.74     yes
Harry Potter and the Sorcerer's Stone  16.25     yes
Angels & Demons                         4.50      no
The Hunger Games                        4.35      no
The Alchemist                          -1.21     yes
The Great Gatsby                       -1.78      no
The Fault in Our Stars                 -2.45      no
The Girl with the Dragon Tattoo        -3.10      no
The Kite Runner                        -3.93      no
Pride and Prejudice                    -6.16      no
The Help                               -6.20      no
The Diary of a Young Girl              -6.44      no
To Kill a Mockingbird                  -7.31      no
1984                                   -8.25      no

mean delta: fantasy 12.26 | other -3.34
none         fantasy wins 49% of mixed pairs, mean D = -2.66


fantasy=yes  fantasy wins 95% of mixed pairs, mean D = +11.83


## Findings (2026-10-12, qwen-7B, jobs 1447314 / 1447317)

**No bugs.** Both runs complete (910 rows), the prompt prefix is right, all 5 templates agree
(Spearman 0.95 / 0.96). Saturation is lower than on laptops (64% / 83% vs 88–99%).

1. **There is a clear preference, and it follows the readers.** Without a contract the model's ranking
   correlates with the Goodreads average rating (Spearman +0.67), weakly with popularity (+0.31), and
   not at all with year (0.00). The guess "the model prefers old classics" was wrong.
2. **Twilight is a strong, stable dislike**: last, it loses 100% of its comparisons on every template
   (mean D −13.2, more than twice the next one). Harry Potter is first (wins 98%).
3. **The contract works**: fantasy wins 49% → 95% of the mixed pairs. Twilight jumps from 14th to 2nd
   (+21.7). The contract overrides even the strongest dislike.
4. **The tags are only an approximation of what the model "thinks" a book is.** The Alchemist is tagged
   fantasy (23% of its genre shelves) but does not move (−1.2). The Hunger Games is not tagged fantasy
   (11%, just under the 12% threshold) but rises (+4.4). The tags are the readers' view, not the model's.
5. **The tags explain little of the baseline preference** (R² 0.20): the choice is about the book,
   not its genre. Under the contract R² rises to 0.78, because the contract is stated in tag terms.
